- **파일명:** `20261020_예측모델제작과제_이름.zip`
    - 과제 수행 파일 (`.ipynb`)
        - 아래 항목이 파일 내에 포함되어 있어야 함
            1. 선택한 문제 번호
            2. 결측치 처리 방법과 이유 (한 줄)
            3. 사용한 피처 목록
            4. 두 모델의 학습 코드와 성능 결과 (목표 달성 여부 표시)
            5. 각 문항에서 "기술하시오"로 요구한 답변 (선택한 문제 기준)
            6. 전체 과정에서 알게 된 점이나 어려웠던 점 (2~3줄)
- **참고 사항**
    - **결과 값이 실수인 경우 소수점 2번째 자리까지 반올림 처리**

## 데이터 설명

포켓몬 **721마리**(1~6세대), **23개 컬럼**으로 이루어져 있습니다.
| 컬럼명 | 설명 |
| --- | --- |
| `Number` | 포켓몬 도감 번호 (고유 ID) |
| `Name` | 포켓몬 이름 |
| `Type_1` | 주 타입 |
| `Type_2` | 보조 타입 (없으면 결측) |
| `Total` | 여섯 능력치(`HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`)의 **합** |
| `HP` | 체력 |
| `Attack` | 물리 공격 |
| `Defense` | 물리 방어 |
| `Sp_Atk` | 특수 공격 |
| `Sp_Def` | 특수 방어 |
| `Speed` | 스피드 |
| `Generation` | 등장 세대 (1~6) |
| `isLegendary` | 전설 포켓몬 여부 (True/False) |
| `Color` | 도감상 색상 |
| `hasGender` | 성별 구분이 있는지 여부 |
| `Pr_Male` | 수컷일 확률 (암컷 확률은 `1 - Pr_Male`) |
| `Egg_Group_1` | 첫 번째 알 그룹 |
| `Egg_Group_2` | 두 번째 알 그룹 (없으면 결측) |
| `hasMegaEvolution` | 메가진화 가능 여부 |
| `Height_m` | 키 (m) |
| `Weight_kg` | 몸무게 (kg) |
| `Catch_Rate` | 포획률 (높을수록 잡기 쉬움) |
| `Body_Style` | 도감상 몸 형태 |

# 💡공통 규칙

- `Number`, `Name`은 식별용 값이므로 **feature로 사용하지 않습니다.**
- 학습/검증 데이터 분리는 `test_size=0.2`, `random_state=42`로 **고정**합니다.
- 모델을 만들 때도 `random_state=42`를 지정합니다.
- 컬럼명은 대소문자까지 정확히 입력하시오. (예: `Pr_Male`)

In [2]:
import platform
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from matplotlib import font_manager
from pprint import pprint

ENCODING = "utf-8-sig"
SEED = 42

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 23)

print("준비 완료")

준비 완료


# 해결 과제 (공통)

### 1. pandas를 사용하여 pokemon_alopez247.csv 파일 데이터를 불러와 DataFrame 으로 저장하시오.

In [3]:
pokemon_df = pd.read_csv("pokemon.csv")

### 2. 저장된 데이터에서 상위 5개 행을 출력하시오.

In [4]:
pokemon_df.head()

,Number,Name,Type_1,Type_2,Total,HP,Attack,Defense,Sp_Atk,Sp_Def,Speed,Generation,isLegendary,Color,hasGender,Pr_Male,Egg_Group_1,Egg_Group_2,hasMegaEvolution,Height_m,Weight_kg,Catch_Rate,Body_Style
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False,Green,True,0.875,Monster,Grass,False,0.71,6.9,45,quadruped
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False,Green,True,0.875,Monster,Grass,False,0.99,13.0,45,quadruped
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False,Green,True,0.875,Monster,Grass,True,2.01,100.0,45,quadruped
3,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False,Red,True,0.875,Monster,Dragon,False,0.61,8.5,45,bipedal_tailed
4,5,Charmeleon,Fire,NaN,405,58,64,58,80,65,80,1,False,Red,True,0.875,Monster,Dragon,False,1.09,19.0,45,bipedal_tailed


### 3. 각 열의 이름, 결측치 여부, 데이터 타입(`dtype`)을 한 번에 확인하시오.

<aside>

`df.info()`를 실행한 후, 출력 결과를 바탕으로 다음을 기술하시오.

- 결측치가 존재하는 열과 결측 개수
- 각 열의 결측이 **무엇을 의미하는지** (위 데이터 설명 참고)
</aside>

In [5]:
pokemon_df.info()
print()

for col in pokemon_df.columns:
    if pokemon_df[col].isna().sum() > 0:
        print(f"{col} 컬럼 결측: {pokemon_df[col].isna().sum()}개")

<class 'pandas.DataFrame'>
RangeIndex: 721 entries, 0 to 720
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Number            721 non-null    int64  
 1   Name              721 non-null    str    
 2   Type_1            721 non-null    str    
 3   Type_2            350 non-null    str    
 4   Total             721 non-null    int64  
 5   HP                721 non-null    int64  
 6   Attack            721 non-null    int64  
 7   Defense           721 non-null    int64  
 8   Sp_Atk            721 non-null    int64  
 9   Sp_Def            721 non-null    int64  
 10  Speed             721 non-null    int64  
 11  Generation        721 non-null    int64  
 12  isLegendary       721 non-null    bool   
 13  Color             721 non-null    str    
 14  hasGender         721 non-null    bool   
 15  Pr_Male           644 non-null    float64
 16  Egg_Group_1       721 non-null    str    
 17  Egg_Grou

---
#### 결측이 존재하는 열과 결측 개수
- 결측이 존재하는 열: `Type_2`, `Pr_Male`, `Egg_Group_2`
- 결측 개수: `Type_2 = 371`, `Pr_Male = 77`, `Egg_Group_2 = 530`

#### 결측의 의미
- `Type_2`: 보조 타입이 없는 포켓몬(주 타입만 있음)
- `Pr_Male`: 성별 구분이 없는 포켓몬(`hasGender` 값이 False). 성별 구분이 없으므로 수컷일 확률도 의미가 없다.
- `Egg_Group_2`: 두 번째 알 그룹이 없는 포켓몬 (포켓몬 교배 시스템)
---

### 4. 결측치를 처리하시오.

- 처리 방식은 자유롭게 선택해도 됩니다. (예: `fillna()`로 `'None'` 또는 특정 숫자 채우기)
- **왜 그렇게 처리했는지 한 줄로 기술**하시오.
- 처리 후 전체 결측치 개수가 0인지 확인하시오.

In [6]:
pokemon_df["Type_2"] = pokemon_df["Type_2"].fillna("None")
pokemon_df["Pr_Male"] = pokemon_df["Pr_Male"].fillna(-1)
pokemon_df["Egg_Group_2"] = pokemon_df["Egg_Group_2"].fillna("None")

print(f"총 {len(pokemon_df.columns)}열 결측 개수\n\n"
      f"{pokemon_df.isna().sum()}")

총 23열 결측 개수

Number              0
Name                0
Type_1              0
Type_2              0
Total               0
HP                  0
Attack              0
Defense             0
Sp_Atk              0
Sp_Def              0
Speed               0
Generation          0
isLegendary         0
Color               0
hasGender           0
Pr_Male             0
Egg_Group_1         0
Egg_Group_2         0
hasMegaEvolution    0
Height_m            0
Weight_kg           0
Catch_Rate          0
Body_Style          0
dtype: int64


---
#### 결측치 처리 방법과 이유
- `Type_2`: `"None"`으로 처리. 보조 타입이 없는 포켓몬이므로 `값이 없음`을 나타내기 위해 `"None"`으로 처리하였다. (따옴표 없이 `None`으로 처리하면 결측치가 그대로 남게 되므로 문자열 형태로 처리함.)
- `Pr_Male`: `-1`로 처리. 수컷일 확률을 `0`으로 하면 암컷일 확률이 `1`이 된다. 이 경우 암컷만 있는 포켓몬과 구별할 수 없다. 따라서 확률에 존재할 수 없는 `-1`로 처리하여 로직 작성 시 `if`문 등으로 걸러낼 수 있도록 한다.
- `Egg_Group_2`: `"None"`으로 처리. 미발견(`Undiscovered`)으로 처리할 경우, 교배 불가 포켓몬이 된다. 단일 그룹(`Egg_Group_1`값만 존재하는 그룹) 포켓몬임을 명시하기 위해 `"None"`으로 처리한다.
---

### 5. 전설 포켓몬(isLegendary=True)과 일반 포켓몬이 각각 몇 마리인지, 전설 포켓몬의 비율(%)은 얼마인지 계산하시오.

In [7]:
legendary_count = pokemon_df["isLegendary"].sum()
normal_count = pokemon_df.shape[0] - legendary_count
legendary_ratio = 100 * legendary_count / pokemon_df.shape[0]

print(f"전설 포켓몬: {legendary_count}마리\n"
      f"일반 포켓몬: {normal_count}마리\n"
      f"전설 포켓몬 비율(%): {legendary_ratio:.2f}%")

전설 포켓몬: 46마리
일반 포켓몬: 675마리
전설 포켓몬 비율(%): 6.38%


## 해결 과제 (선택)

> 아래 **문제 [1]** 또는 **문제 [2]** 중 **하나만 선택**하여 풀이하시오.
> 
## 선택 문항: [1]

## 문제 [1] 포켓몬 능력치(`Total`) 예측 (회귀)

능력치 이외의 정보(타입, 세대, 키, 몸무게 등)로 `Total`을 예측하는 회귀 모델을 만든다.

### [1-1] `Total`을 타깃(`y`)으로 하고, 사용할 feature(`X`)를 정하시오.

<aside>

#### `Total`과 여섯 능력치(`HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`)는 feature로 사용하지 않는다.

`Total`은 여섯 능력치를 더한 값이다. 능력치를 feature에 넣으면 정답이 입력에 그대로 들어가는 것이므로(데이터 누수) 예측 모델이라고 할 수 없다.

---

**사용 가능한 컬럼:** `Type_1`, `Type_2`, `Generation`, `isLegendary`, `Color`, `hasGender`, `Pr_Male`, `Egg_Group_1`, `Egg_Group_2`, `hasMegaEvolution`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Body_Style`

</aside>

In [8]:
FEATURES = ["Type_1", "Type_2", "Generation", "isLegendary", "Color",
            "hasGender", "Pr_Male", "Egg_Group_1", "Egg_Group_2", "hasMegaEvolution",
            "Height_m", "Weight_kg", "Catch_Rate", "Body_Style"]

y = pokemon_df["Total"]

### [1-2] 범주형 컬럼을 숫자로 변환하시오.

- `pd.get_dummies()` 등을 사용해도 됩니다.
- 변환 후 `X`의 컬럼 개수를 출력하시오.

In [16]:
# dtype이 str인 컬럼들
STR_COLS = ["Type_1", "Type_2", "Color", "Egg_Group_1", "Egg_Group_2", "Body_Style"]

# One-Hot으로 할지 결정하기 위해 고유값 개수 확인
for col in STR_COLS:
    print(f"{col}: {pokemon_df[col].nunique()}개")
# Type_1 = 18, Type_2 = 19, Color = 10, Egg_Group_1 = 15, Egg_Group_2 = 14 Body_Style = 14
# 20개 미만이므로 One-Hot 인코딩

X = pd.get_dummies(pokemon_df[FEATURES])

# 바뀐 컬럼을 확인하기 위해 X.columns 출력
# print(f"One-Hot 인코딩 후 X의 컬럼: \n{X.columns}\n")
# bool, float, int형이 아닌 범주형 컬럼만 인코딩 되었는지 확인 완료
print(f"X의 컬럼 수: {X.shape[1]}열")

Type_1: 18개
Type_2: 19개
Color: 10개
Egg_Group_1: 15개
Egg_Group_2: 14개
Body_Style: 14개
X의 컬럼 수: 97열


### [1-3] 수치형 변수(`Generation`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Pr_Male`)와 `Total`의 상관계수를 구하시오.

- `df[[...]].corr()['Total']` 형식으로 계산하시오.
- 상관계수 절댓값이 가장 큰 변수 3개를 기술하시오.

In [10]:
corr_matrix = (pokemon_df[["Generation", "Height_m", "Weight_kg", 
                           "Catch_Rate", "Pr_Male", "Total"]].corr()["Total"])
corr_sorted = corr_matrix.sort_values(key=lambda x: x.abs(), ascending=False)

print(corr_sorted.round(2))

Total         1.00
Catch_Rate   -0.74
Weight_kg     0.54
Height_m      0.53
Pr_Male      -0.32
Generation    0.09
Name: Total, dtype: float64


### |상관계수| TOP 3
- Catch_Rate 컬럼: -0.74
- Weight_kg 컬럼: 0.54
- Height_m 컬럼: 0.53

> 포획 확률은 높은 상관관계를 보이고, 키와 체중도 0.5 이상의 상관계수를 보인다. 수컷일 확률과의 상관계수는 -0.32로 다른 컬럼들에 비해 높지는 않지만 아예 상관이 없는 것도 아닌 것으로 보인다. 세대와의 상관계수는 0.09로 0에 가깝다. 즉, 세대는 Total을 예측하기 위한 유의미한 피처로 쓰기에는 부적절하다. => FEATURES에서 삭제.

In [11]:
FEATURES = ["Type_1", "Type_2", "isLegendary", "Color", "hasGender", 
            "Pr_Male", "Egg_Group_1", "Egg_Group_2", "hasMegaEvolution",
            "Height_m", "Weight_kg", "Catch_Rate", "Body_Style"]

### [1-4] 학습 데이터와 검증 데이터를 분리하고, 각각의 크기(`shape`)를 출력하시오.

- `train_test_split(X, y, test_size=0.2, random_state=42)`

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=SEED)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print()

# train : test = 0.8 : 0.2이므로 얼추 4배가 나오는지 확인
print(f"X_train / X_test = {round(X_train.shape[0] / X_test.shape[0], 1)}")

X_train: (576, 98), X_test: (145, 98)
y_train: (576,), y_test: (145,)

X_train / X_test = 4.0


### [1-5] `LinearRegression` 모델을 학습시키고, 검증 데이터에 대한 **R²**와 **RMSE**를 계산하시오.

<aside>

#### 평가 지표

| 지표 | 의미 |
| --- | --- |
| R² | 모델이 값의 변동을 얼마나 설명하는지. 1에 가까울수록 좋음 |
| RMSE | 평균적으로 얼마나 빗나갔는지. `Total` 점수와 같은 단위이며 작을수록 좋음 |
- `r2_score`, `mean_squared_error` (RMSE는 제곱근을 취하여 계산, `root_mean_squared_error` 활용 가능)
- 참고: `Total`의 표준편차는 약 110이다. 평균값만 예측하면 RMSE가 약 110, R²가 0 근처이다.
</aside>

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# 선형 회귀에 스케일링은 필수가 아니지만 계수 크기 비교를 해보기 위해
# StandardScaler를 적용하였습니다.
def make(model):
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", model)
    ])

linear_reg = make(LinearRegression())

linear_reg.fit(X_train, y_train)

# linear_Reg.named_steps["m"]: 절편과 계수 확인을 위해 모델 객체 추출
# linear_Reg.coef_: 가중치(계수) 배열
coef = pd.Series(linear_reg.named_steps["m"].coef_, index=X_train.columns).sort_values()
intercept = linear_reg.named_steps["m"].intercept_

print(f"  * 절편 b = {intercept:.2f}\n")
print(f"  * 가중치 w (상위 10개)")
print(coef.head(10).round(2).to_string())
print()

pred = linear_reg.predict(X_test)

print(f"R²   = {r2_score(y_test, pred):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.2f}")

  * 절편 b = 420.14

  * 가중치 w (상위 10개)
Catch_Rate                   -63.54
Type_1_Bug                   -16.78
Egg_Group_1_Undiscovered     -12.53
Pr_Male                      -11.69
Body_Style_serpentine_body    -8.05
Type_2_Fairy                  -7.46
Color_Blue                    -6.84
Egg_Group_1_Amorphous         -6.22
Egg_Group_2_Water_2           -5.74
Egg_Group_1_Monster           -5.68

R²   = 0.63
RMSE = 66.87


### [1-6] RandomForestRegressor 모델을 학습시키고, 검증 데이터에 대한 R²와 RMSE를 계산하시오.

In [14]:
from sklearn.ensemble import RandomForestRegressor

random_reg = RandomForestRegressor(n_estimators=100, max_depth=6,
                                   random_state=SEED, n_jobs=-1)

random_reg.fit(X_train, y_train)
pred_random = random_reg.predict(X_test)

print(f"R²   = {r2_score(y_test, pred_random):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, pred_random)):.2f}")

R²   = 0.78
RMSE = 51.11


### [1-7] 두 모델의 성능을 표로 정리하고, 성능 목표 달성 여부를 확인하시오.

<aside>

#### 성능 목표

| 모델 | R² | RMSE |
| --- | --- | --- |
| `LinearRegression` | 0.60 이상 | 70 이하 |
| `RandomForestRegressor` | 0.75 이상 | 55 이하 |
</aside>

- 두 모델 중 어느 쪽이 더 잘 맞았는지, 그 이유는 무엇일지 기술하시오.

---
### 두 모델의 성능 비교 표
|모델|R²|RMSE|목표 달성 여부|
|---|---|---|---|
|`LinearRegression`|0.63|66.87|달성|
|`RandomForestRegressor|0.78|51.11|달성|

### 더 잘 맞힌 모델과 그 이유
`RandomForestRegressor`: `LinearRegression`은 변수들 간의 단순한 선형 관계만을 학습할 수 있다. 이에 반해 `RandomForestRegressor`는 여러 결정 트리가 결합한 앙상블 모델로서, 트리들이 예측한 값의 **평균**으로 수치를 결정한다. 비선형 관계도 잡아낼 수 있으므로 `LinearRegression`보다 좋은 성능을 보인다. `RandomForestRegression`은 외삽 불가라는 특징도 가지고 있지만, 게임은 보통 밸런스를 위해 기존 능력치에서 크게 벗어나지 않도록 설계하므로 `LinearRegression`보다 모델로서 적합하다.

### [1-8] (선택) 확인 실험

- 여섯 능력치를 feature에 추가하여 `LinearRegression`을 다시 학습시키고, R²가 왜 그렇게 나오는지 기술하시오. (제출 모델과 별도의 셀에서 진행)
- `Catch_Rate`를 feature에서 제외하고 두 모델의 성능이 어떻게 달라지는지 확인하시오.

In [15]:
# 여섯 능력치를 피처로 추가하여 다시 학습
EX_FEATURES = FEATURES + ["HP", "Attack", "Defense", "Sp_Atk", "Sp_Def", "Speed"]

ex_model = make(LinearRegression())
X_ex = pd.get_dummies(pokemon_df[EX_FEATURES])
# X_ex에 맞는 정답지를 만들기 위해 train_test_split을 다시 호출
# 기존 y_train, y_test는 FEATURES에 맞는 순서 번호를 가지므로 다시 섞어야 한다.
X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(X_ex, y, test_size=0.2,
                                                                random_state=SEED)

ex_model.fit(X_train_ex, y_train_ex)
ex_pred = ex_model.predict(X_test_ex)

print("여섯 능력치를 피처에 추가한 후 모델의 성능 확인")
print(f"R²   = {r2_score(y_test_ex, ex_pred):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test_ex, ex_pred)):.2f}")
print()

# FEATURES에서 "Catch_Rate" 컬럼 제외하고 성능 비교하기
FEATURES_WITHOUT_CR = FEATURES.copy()
FEATURES_WITHOUT_CR.remove("Catch_Rate")

X_ex2 = pd.get_dummies(pokemon_df[FEATURES_WITHOUT_CR])

X_tr, X_te, y_tr, y_te = train_test_split(X_ex2, y, test_size=0.2,
                                          random_state=SEED)

ex_lin = make(LinearRegression())
ex_ran = RandomForestRegressor(n_estimators=100, max_depth=6,
                               random_state=SEED, n_jobs=-1)
ex_lin.fit(X_tr, y_tr)
ex_ran.fit(X_tr, y_tr)
ex_lin_pred, ex_ran_pred = ex_lin.predict(X_te), ex_ran.predict(X_te)

print("'Catch_Rate' 컬럼을 제외하여 학습시킨 모델 - LinearRegression")
print(f"R²   = {r2_score(y_te, ex_lin_pred):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, ex_lin_pred)):.2f}")
print()

print("'Catch_Rate' 컬럼을 제외하여 학습시킨 모델 - RandomForestRegressor")
print(f"R²   = {r2_score(y_te, ex_ran_pred):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_te, ex_ran_pred)):.2f}")

여섯 능력치를 피처에 추가한 후 모델의 성능 확인
R²   = 1.00
RMSE = 0.00

'Catch_Rate' 컬럼을 제외하여 학습시킨 모델 - LinearRegression
R²   = 0.38
RMSE = 86.92

'Catch_Rate' 컬럼을 제외하여 학습시킨 모델 - RandomForestRegressor
R²   = 0.67
RMSE = 62.99


## 이런 결과가 나오는 이유
1. 여섯 능력치를 피처에 추가했을 때의 성능
    - R² = 1, RMSE = 0으로 완벽한 성능을 보인다. Total의 값은 여섯 능력치의 합으로 정답을 알려준 셈이기 때문이다.
    - 데이터 누수로 인한 과적합 상태이다.
    - **합**이란 제곱이나 삼각함수 등 비선형 관계가 포함되지 않은 완벽한 선형 관계이므로 비선형 관계로 인해 놓치는 정답 또한 없기 때문에 성능이 줄어들 일이 전혀 없다.
2. Catch_Rate 컬럼을 제외하여 학습시킨 모델
    - LinearRegression
        - R²   : 0.63  -> 0.38
        - RMSE : 66.87 -> 86.92
    - RandomForestRegressor
        - R²   : 0.78  -> 0.67
        - RMSE : 51.11 -> 62.99
    - [1-3]에서 상관계수를 구했을 때 'Total' 컬럼의 값은 'Catch_Rate' 컬럼과 가장 높은 상관관계(-0.74, 절댓값이 0.74)를 보였다. 잡기 어려울수록 'Total' 컬럼의 값이 높은 것으로 해석할 수 있다. 결과 예측에 가장 적합했던 피처를 제외하였으므로 두 모델 모두 성능이 급감한다.

---
### 전체 과정에서 알게 된 점과 어려웠던 점
- train_test_split() 함수 호출 시 random_state를 고정해야 여러 번 실행해도 결과가 동일하다.
- One-Hot인코딩을 먼저 하고 훈련/검증용 데이터를 분리해야 컬럼 수 차이로 인한 오류가 발생하지 않는다.
- `pd.get_dummies(df[FEATURES])`로 모든 범주형 컬럼에 대해 One-Hot 인코딩을 수행할 수 있다.

---

## 문제 [2] 전설 포켓몬 여부(`isLegendary`) 예측 (분류)

능력치와 속성 정보로 포켓몬이 전설인지 아닌지 맞히는 분류 모델을 만든다.

### [2-1] `isLegendary`를 타깃(`y`)으로 하고, 사용할 feature(`X`)를 정하시오.

- `isLegendary`를 `True`는 1, `False`는 0이 되도록 정수로 변환하시오.

<aside>

#### ⚠️`Total`은 여섯 능력치의 합으로 중복 정보이므로 feature로 사용하지 않는다.

**사용 가능한 컬럼** : `HP`, `Attack`, `Defense`, `Sp_Atk`, `Sp_Def`, `Speed`, `Type_1`, `Type_2`, `Generation`, `Color`, `hasGender`, `Pr_Male`, `Egg_Group_1`, `Egg_Group_2`, `hasMegaEvolution`, `Height_m`, `Weight_kg`, `Catch_Rate`, `Body_Style`

</aside>

### [2-2] 모든 포켓몬을 **"전설 아님(0)"** 으로 예측했을 때의 **accuracy**와 **F1**을 계산하시오.

- `f1_score(y, pred, zero_division=0)`
- 이 값이 기준선(baseline)이 된다. 이후 모델의 성능은 이 기준선과 비교하여 해석하시오.
- 이 결과를 보고, 이 문제에서 accuracy만으로 모델을 평가하면 안 되는 이유를 기술하시오.

### [2-3] 범주형 컬럼을 숫자로 변환하시오.

- `pd.get_dummies()` 등을 사용해도 됩니다.

### [2-4] 학습 데이터와 검증 데이터를 분리하고, 각각에 포함된 전설 포켓몬 수를 출력하시오.

- `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`
- `stratify=y`를 지정하면 학습/검증 데이터의 전설 비율이 전체와 비슷하게 유지된다.

### [2-5] `LogisticRegression` 모델을 학습시키고, 검증 데이터에 대한 **accuracy**, **F1**(전설=1 기준), **confusion matrix**를 구하시오.

- 숫자의 크기 차이에 민감한 모델이므로 `StandardScaler`로 스케일링한 값을 사용하는 것을 권장한다. (스케일러는 학습 데이터로만 `fit` 한다.)
- 학습이 끝나지 않는다는 경고가 나오면 `max_iter=1000` 등으로 늘려 보시오.

<aside>

#### 평가 지표

| 지표 | 의미 |
| --- | --- |
| accuracy | 전체 중 맞힌 비율. 클래스 비율이 치우치면 믿기 어려움 |
| precision | 모델이 전설이라고 한 것 중 진짜 전설인 비율 |
| recall | 진짜 전설 중 모델이 찾아낸 비율 |
| F1 | precision과 recall을 함께 보는 값. 둘 다 높아야 높음 |
| confusion matrix | 맞힌 것과 틀린 것을 종류별로 센 표 |
</aside>

### [2-6] RandomForestClassifier 모델을 학습시키고, 검증 데이터에 대한 accuracy, F1, confusion matrix를 구하시오.

### [2-7] 두 모델의 confusion matrix에서 전설 클래스(1)의 **오분류 패턴**을 분석하시오.

- "전설인데 아니라고 한 경우"와 "전설이 아닌데 전설이라고 한 경우"의 개수를 각각 기술하시오.
- (선택) 잘못 분류된 포켓몬의 이름과 능력치를 출력하여, 어떤 특징이 있는지 기술하시오.

### [2-8] 두 모델의 성능을 표로 정리하고, 성능 목표 달성 여부를 확인하시오.

<aside>

#### 성능 목표

| 모델 | accuracy | F1 (전설=1) |
| --- | --- | --- |
| `LogisticRegression` | 0.95 이상 | 0.60 이상 |
| `RandomForestClassifier` | 0.95 이상 | 0.60 이상 |
</aside>

- 2-2에서 구한 기준선과 비교하여, 모델이 의미 있게 학습했는지 기술하시오.
- 두 모델 중 어느 쪽이 더 좋았는지, 그 이유는 무엇일지 기술하시오.